In [1]:
from IPython.display import clear_output
import os

!git clone --branch shantanu https://github.com/AISC-Linear-Probe-Gen/Probe-Generalisation.git
%pip install --upgrade mech-interp-toolkit
clear_output()


os.chdir("/content/Probe-Generalisation/research/obfuscated_activations")

In [ ]:
# Pin transformers to a version compatible with nnsight.
# transformers>=5.0 added @check_model_inputs which requires **kwargs in LlamaModel.forward,
# breaking nnsight's tracer.invoke() call.
!pip install "transformers<5.0" --quiet
from IPython.display import clear_output
clear_output()

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Standard library imports
from pathlib import Path
import joblib
import re
from collections import defaultdict

# Third-party imports
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm_notebook as tqdm

# mech_interp_toolkit imports
from mech_interp_toolkit.gradient_based_attribution import simple_ig_with_probes
from mech_interp_toolkit.activation_utils import get_activations, get_embeddings_dict, concat_activations
from mech_interp_toolkit.linear_probes import LinearProbe
from mech_interp_toolkit.utils import load_model_tokenizer_config, set_global_seed

from datasets import load_dataset
from utils.data import extract_user_instruction
import einops

In [15]:
dataset_name = "Mechanistic-Anomaly-Detection/llama3-jailbreaks"
split = "circuit_breakers_test"
model_name = "meta-llama/Llama-3.2-3B-Instruct"
suffix_path = "ra_suffix.pt"
probe_path = Path("t_probes")

batch_size = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_global_seed(0)
torch.set_grad_enabled(True)

torch.autograd.grad_mode.set_grad_enabled(mode=True)

In [5]:
class QuickScaler:
    def __init__(self, scaler) -> None:
        self.scale = scaler.scale_
        self.mean = scaler.mean_

    def __call__(self, x) -> torch.Tensor:
        return (x-self.mean)/self.scale


In [9]:
dataset = load_dataset(dataset_name, split=split)
prompts_str = [extract_user_instruction(x) for x in dataset["prompt"]]  # type: ignore
clear_output()

model, ch_tokenizer, config = load_model_tokenizer_config(
    model_name,
    suffix="",
    system_prompt="",
    attn_type="sdpa",
)
clear_output()


def load_suffix(suffix_path: str, device: torch.device) -> torch.Tensor:
    if not suffix_path.endswith(".pt"):
        raise ValueError("suffix_path must be a .pt embedding file")
    suffix_emb = torch.load(suffix_path, map_location=device)
    if suffix_emb.dim() == 2:
        suffix_emb = suffix_emb.unsqueeze(0)
    return suffix_emb


n_layers = config.num_hidden_layers
components = [(i, "layer_out") for i in range(n_layers)]

suffix_embed = load_suffix(suffix_path, device)
len_suffix = suffix_embed.shape[1]

In [10]:
probe_dict = {}
for f in probe_path.glob("*.joblib"):
    layer = int(re.findall(r"layer_(\d+)", str(f))[0])
    probe = joblib.load(f)

    lp = LinearProbe("classification")
    lp.location = (layer, "layer_out")
    lp.weight = probe["model"].coef_
    lp.bias = probe["model"].intercept_
    lp.scaler = QuickScaler(probe["scaler"])  # type:ignore

    probe_dict[layer] = lp

In [11]:
def metric_fn(x):
    return x[:,-1,:]

outputs = defaultdict(list)

In [16]:
for layer in range(n_layers):

    for batch_start in tqdm(range(0, len(prompts_str), batch_size), desc=str(f"layer: {layer}")):
        batch_prompts = prompts_str[batch_start : batch_start + batch_size]

        batch_dict = ch_tokenizer(prompts=batch_prompts)
        batch_embeds_dict = get_embeddings_dict(model, batch_dict)
        batch_embeds = batch_embeds_dict["inputs_embeds"]
        batch_attn_mask = batch_embeds_dict["attention_mask"]

        batch_suffix = einops.repeat(
                suffix_embed,
                "1 pos d_model -> b pos d_model",
                b=len(batch_prompts),
            )
        new_embeds_dict = {
            "inputs_embeds": torch.cat(
                [batch_embeds[:, :-5, :], batch_suffix, batch_embeds[:, -5:, :]], dim=1
            ),
            "attention_mask": torch.cat(
                [batch_attn_mask, torch.ones(len(batch_prompts), len_suffix, device=batch_attn_mask.device)],
                dim=1,
            ),
        }

        baseline = {
            "inputs_embeds": torch.zeros_like(new_embeds_dict["inputs_embeds"]),
            "attention_mask": torch.cat(
                [batch_attn_mask, torch.ones(len(batch_prompts), len_suffix, device=batch_attn_mask.device)],
                dim=1,
            ),
        }

        ig_scores = simple_ig_with_probes(
            model=model,
            input_dict=new_embeds_dict,
            baseline_dict=baseline,
            probe=probe_dict[layer]
        )

        outputs[layer].append(ig_scores)

layer: 0:   0%|          | 0/32 [00:00<?, ?it/s]

NNsightException: 

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1881, in _call_impl
    return inner()
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1829, in inner
    result = forward_call(*args, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py", line 835, in wrapper
    output = func(self, *args, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/llama/modeling_llama.py", line 486, in forward
    outputs: BaseModelOutputWithPast = self.model(
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/mod

In [14]:
new_embeds_dict

{'inputs_embeds': tensor([[[-1.5137e-02,  1.0071e-02, -4.9438e-03,  ...,  3.7689e-03,
           -1.1475e-02, -2.3346e-03],
          [-1.5137e-02,  1.0071e-02, -4.9438e-03,  ...,  3.7689e-03,
           -1.1475e-02, -2.3346e-03],
          [-1.5137e-02,  1.0071e-02, -4.9438e-03,  ...,  3.7689e-03,
           -1.1475e-02, -2.3346e-03],
          ...,
          [ 1.5442e-02, -2.3193e-02,  1.5869e-02,  ..., -2.5940e-03,
            5.8594e-03, -6.0791e-02],
          [-1.1658e-02,  6.1646e-03,  5.2185e-03,  ...,  9.5825e-03,
           -1.6708e-03, -1.6235e-02],
          [-6.5308e-03, -6.9809e-04, -6.7902e-04,  ...,  4.3869e-04,
            1.4038e-02,  1.0834e-03]],
 
         [[-1.5137e-02,  1.0071e-02, -4.9438e-03,  ...,  3.7689e-03,
           -1.1475e-02, -2.3346e-03],
          [-1.5137e-02,  1.0071e-02, -4.9438e-03,  ...,  3.7689e-03,
           -1.1475e-02, -2.3346e-03],
          [-1.5137e-02,  1.0071e-02, -4.9438e-03,  ...,  3.7689e-03,
           -1.1475e-02, -2.3346e-03],
  

In [ ]:
import pickle

with open("sig_scores.pkl", "wb") as f:
    pickle.dump(outputs, f)